<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">هیچ عددی نباید به <bdi dir="ltr">Head</bdi> اشتباه برود</h1>
<p style="text-align:right">درس 41 از 92 · چگونه <bdi dir="ltr">C</bdi> ویژگی را میان چند سر تقسیم کنیم؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">35-split-heads</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/35-split-heads.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">تقسیم ویژگی‌ها را با نشانی عناصر، نه فقط <bdi dir="ltr">Shape</bdi>، تأیید کنید.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">reshape</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">transpose</code> و رابطهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">C=H*D</code> را مرور کنید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۴۵–۸۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x</code> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(2,5,12)</code> با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">H=3</code>، عنصر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">x[1,3,9]</code> بعد از تقسیم در کدام <bdi dir="ltr">Head</bdi> و کدام ویژگی آن است؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
x = torch.arange(120.).reshape(2,5,12)
print('tracked value:',x[1,3,9].item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">split_heads(x, heads)</code> را بنویسید: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,C)</code> به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,D)</code>. اگر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">C</code> بر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">H</code> بخش‌پذیر نیست یا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">H</code> مثبت نیست، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ValueError</code> بدهید. داده را حذف یا با صفر پر نکنید.</p>
</div>

In [ ]:
def split_heads(x, heads):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = split_heads(x,3)
    if result is None: return False
    assert result.shape == (2,3,5,4)
    assert result[1,2,3,1] == x[1,3,9]
    for H in (1,2,4):
        out = split_heads(x,H); D = 12//H
        for h in range(H):
            assert torch.equal(out[:,h],x[:,:,h*D:(h+1)*D])
    for invalid in (0,-1,5):
        try: split_heads(x,invalid)
        except ValueError: pass
        else: raise AssertionError('reject invalid C/H')
    attention = CausalSelfAttention(ModelConfig(12,8,12,3,1,0.)).eval()
    trace = {}; attention(x,trace=trace)
    q = attention.qkv(x).chunk(3,-1)[0]
    torch.testing.assert_close(split_heads(q,3),trace['q'])
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">H</code> را میان ۱، ۳ و ۴ تغییر دهید و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">C=12</code> را ثابت نگه دارید. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">D</code> و تعداد جدول‌های <bdi dir="ltr">Attention</bdi> چه می‌شوند؟</p>
</div>

In [ ]:
for H in (1,3,4):
    config = ModelConfig(12,8,12,H,1,0.)
    module = CausalSelfAttention(config)
    print('H,D,parameters:',H,12//H,sum(p.numel() for p in module.parameters()))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">reshape</code> مستقیم به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,H,T,D)</code> <bdi dir="ltr">Shape</bdi> درست ولی نشانی اشتباه می‌سازد. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">repair_split(x, heads)</code> را اصلاح کنید؛ ورودی‌های این بخش تقسیم‌پذیرند.</p>
</div>

In [ ]:
wrong = x.reshape(2,3,5,4)
print('wrong value:',wrong[1,2,3,1].item(),'expected:',x[1,3,9].item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def repair_split(x, heads):
    # TODO
    return None

In [ ]:
def test_repair():
    result = repair_split(x,3)
    if result is None: return False
    assert torch.equal(result[:,1],x[:,:,4:8])
    y = torch.arange(48.).reshape(1,4,12)
    assert torch.equal(repair_split(y,2)[:,1],y[:,:,6:])
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">تابع شما با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">q</code> ثبت‌شده در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace</code> واقعی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CausalSelfAttention</code> مقایسه شد. تقسیم پس از <bdi dir="ltr">Projection</bdi> انجام می‌شود؛ هر <bdi dir="ltr">Head</bdi> همهٔ موقعیت‌های مجاز را می‌بیند، نه یک قطعهٔ متن جدا.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام <bdi dir="ltr">assertion</bdi> می‌توانست خطایی را بگیرد که آزمون <bdi dir="ltr">Shape</bdi> از آن عبور می‌کرد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/35-split-heads.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/35-split-heads.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>